# Import library

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import top_k_accuracy_score
from jcopml.utils import save_model
from jcopml.utils import load_model

# Data preprocessing

In [2]:
# read data
df = pd.read_csv('Dataset_Discoaster(Tree)-Neogene-Resample.csv')
df

,Jumlah_Lengan,Bercabang,Simetris,Knob,Ukuran_Lengan,Bentuk_Lengan,Ujung_Lengan,Ujung_Lengan_Melengkung,Spesies
0,3,Ya,Ya,Tidak,Panjang,Pipih,Lancip,Tidak,Discoaster triradiatus
1,4,Tidak,Ya,Tidak,Pendek,Pipih,Tumpul,Tidak,Discoaster tamalis
2,4,Tidak,Tidak,Tidak,Pendek,Pipih,Lancip,Tidak,Discoaster alyssae
3,5,Tidak,Tidak,Ya,Panjang,Pipih,Lancip,Tidak,Discoaster asymmetricus
4,5,Ya,Tidak,Tidak,Panjang,Pipih,Lancip,Tidak,Discoaster pentaradiatus
5,5,Ya,Tidak,Tidak,Panjang,Tebal,Lancip,Tidak,Discoaster prepentaradiatus
6,5,Tidak,Ya,Tidak,Panjang,Pipih,Lancip,Ya,Discoaster hamatus
7,5,Tidak,Ya,Tidak,Pendek,Pipih,Tumpul,Tidak,Discoaster bellus
8,5,Tidak,Ya,Ya,Panjang,Pipih,Tumpul,Tidak,Discoaster quinqueramus
9,5,Tidak,Ya,Ya,Pendek,Pipih,Tumpul,Tidak,Discoaster berggrenii


In [3]:
df.shape

(55, 9)

In [4]:
# feautures encoding
ordl_encoder = preprocessing.OrdinalEncoder(dtype=int)
ordl_encoder.fit(df[['Bercabang', 'Simetris', 'Knob', 'Ukuran_Lengan', 'Bentuk_Lengan', 'Ujung_Lengan', 'Ujung_Lengan_Melengkung']])
df[['Bercabang', 'Simetris', 'Knob', 'Ukuran_Lengan', 'Bentuk_Lengan', 'Ujung_Lengan', 'Ujung_Lengan_Melengkung']] = ordl_encoder.transform(df[['Bercabang', 'Simetris', 'Knob', 'Ukuran_Lengan', 'Bentuk_Lengan', 'Ujung_Lengan', 'Ujung_Lengan_Melengkung']])

In [5]:
df.head()

,Jumlah_Lengan,Bercabang,Simetris,Knob,Ukuran_Lengan,Bentuk_Lengan,Ujung_Lengan,Ujung_Lengan_Melengkung,Spesies
0,3,1,1,0,0,0,0,0,Discoaster triradiatus
1,4,0,1,0,1,0,1,0,Discoaster tamalis
2,4,0,0,0,1,0,0,0,Discoaster alyssae
3,5,0,0,1,0,0,0,0,Discoaster asymmetricus
4,5,1,0,0,0,0,0,0,Discoaster pentaradiatus


In [6]:
# target label encoding
lbl_encoder = preprocessing.LabelEncoder()
lbl_encoder.fit(df['Spesies'])
df['Spesies'] = lbl_encoder.transform(df['Spesies'])

In [7]:
df.head()

,Jumlah_Lengan,Bercabang,Simetris,Knob,Ukuran_Lengan,Bentuk_Lengan,Ujung_Lengan,Ujung_Lengan_Melengkung,Spesies
0,3,1,1,0,0,0,0,0,34
1,4,0,1,0,1,0,1,0,33
2,4,0,0,0,1,0,0,0,2
3,5,0,0,1,0,0,0,0,3
4,5,1,0,0,0,0,0,0,24


In [8]:
df.shape

(55, 9)

In [9]:
# check dataset balance
sorted(df['Spesies'].unique())

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36]

In [10]:
print(sorted(Counter(df['Spesies']).items()))

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 10), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 10), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1)]


In [11]:
df['Spesies'].value_counts().describe()

count    37.000000
mean      1.486486
std       2.063191
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      10.000000
Name: Spesies, dtype: float64

In [12]:
# features and targets split
X = df[['Jumlah_Lengan', 'Bercabang', 'Simetris', 'Knob', 'Ukuran_Lengan', 'Bentuk_Lengan', 'Ujung_Lengan', 'Ujung_Lengan_Melengkung']]
X

,Jumlah_Lengan,Bercabang,Simetris,Knob,Ukuran_Lengan,Bentuk_Lengan,Ujung_Lengan,Ujung_Lengan_Melengkung
0,3,1,1,0,0,0,0,0
1,4,0,1,0,1,0,1,0
2,4,0,0,0,1,0,0,0
3,5,0,0,1,0,0,0,0
4,5,1,0,0,0,0,0,0
5,5,1,0,0,0,1,0,0
6,5,0,1,0,0,0,0,1
7,5,0,1,0,1,0,1,0
8,5,0,1,1,0,0,1,0
9,5,0,1,1,1,0,1,0


In [13]:
y = df['Spesies']
y

0     34
1     33
2      2
3      3
4     24
5     27
6     17
7      4
8     29
9      6
10    11
11     5
12     9
13     1
14    10
15     8
16    23
17    13
18    16
19    26
20    32
21    28
22    31
23    35
24    19
25    20
26    15
27     7
28     7
29     7
30     7
31     7
32     7
33     7
34     7
35     7
36     7
37    25
38    30
39    22
40    36
41    21
42    12
43    18
44    14
45    14
46    14
47    14
48    14
49    14
50    14
51    14
52    14
53    14
54     0
Name: Spesies, dtype: int32

In [14]:
# random oversampling
ros = RandomOverSampler(random_state=0)
X_resampled, y_resampled = ros.fit_resample(X, y)

In [15]:
# repeat check dataset balance
X_resampled

,Jumlah_Lengan,Bercabang,Simetris,Knob,Ukuran_Lengan,Bentuk_Lengan,Ujung_Lengan,Ujung_Lengan_Melengkung
0,3,1,1,0,0,0,0,0
1,4,0,1,0,1,0,1,0
2,4,0,0,0,1,0,0,0
3,5,0,0,1,0,0,0,0
4,5,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
365,6,1,1,1,1,0,0,0
366,6,1,1,1,1,0,0,0
367,6,1,1,1,1,0,0,0
368,6,1,1,1,1,0,0,0


In [16]:
y_resampled

0      34
1      33
2       2
3       3
4      24
       ..
365    36
366    36
367    36
368    36
369    36
Name: Spesies, Length: 370, dtype: int32

In [17]:
print(sorted(Counter(y_resampled).items()))

[(0, 10), (1, 10), (2, 10), (3, 10), (4, 10), (5, 10), (6, 10), (7, 10), (8, 10), (9, 10), (10, 10), (11, 10), (12, 10), (13, 10), (14, 10), (15, 10), (16, 10), (17, 10), (18, 10), (19, 10), (20, 10), (21, 10), (22, 10), (23, 10), (24, 10), (25, 10), (26, 10), (27, 10), (28, 10), (29, 10), (30, 10), (31, 10), (32, 10), (33, 10), (34, 10), (35, 10), (36, 10)]


In [18]:
y_resampled.value_counts().describe()

count    37.0
mean     10.0
std       0.0
min      10.0
25%      10.0
50%      10.0
75%      10.0
max      10.0
Name: Spesies, dtype: float64

Train Test Split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=1945, stratify=y_resampled)

In [20]:
print(sorted(Counter(y_train).items()))

[(0, 7), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7), (8, 7), (9, 7), (10, 7), (11, 7), (12, 7), (13, 7), (14, 7), (15, 7), (16, 7), (17, 7), (18, 7), (19, 7), (20, 7), (21, 7), (22, 7), (23, 7), (24, 7), (25, 7), (26, 7), (27, 7), (28, 7), (29, 7), (30, 7), (31, 7), (32, 7), (33, 7), (34, 7), (35, 7), (36, 7)]


In [21]:
y_train.value_counts().describe()

count    37.0
mean      7.0
std       0.0
min       7.0
25%       7.0
50%       7.0
75%       7.0
max       7.0
Name: Spesies, dtype: float64

In [22]:
print(sorted(Counter(y_test).items()))

[(0, 3), (1, 3), (2, 3), (3, 3), (4, 3), (5, 3), (6, 3), (7, 3), (8, 3), (9, 3), (10, 3), (11, 3), (12, 3), (13, 3), (14, 3), (15, 3), (16, 3), (17, 3), (18, 3), (19, 3), (20, 3), (21, 3), (22, 3), (23, 3), (24, 3), (25, 3), (26, 3), (27, 3), (28, 3), (29, 3), (30, 3), (31, 3), (32, 3), (33, 3), (34, 3), (35, 3), (36, 3)]


In [23]:
y_test.value_counts().describe()

count    37.0
mean      3.0
std       0.0
min       3.0
25%       3.0
50%       3.0
75%       3.0
max       3.0
Name: Spesies, dtype: float64

# Algorithm Selection Test

## Support Vector Machine

In [24]:
# regularization parameter
C = [0.1, 1, 10, 100, 1000]
# kernel function
kernel = ['linear', 'poly', 'rbf', 'sigmoid']
# degree of the polynomial kernel function
degree = [2, 3, 4]
# kernel coefficient for 'rbf', 'poly' and 'sigmoid'.
gamma = ['scale', 'auto', 0.1, 1, 10]
# create the params grid
param_grid = {'C': C,
              'kernel': kernel,
               'degree': degree,
             'gamma': gamma}

In [25]:
# GridSearchCV fit
model = SVC(random_state=1)

grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring="accuracy", cv=4, verbose=3)
grid_result = grid.fit(X_train.values, y_train.values)

Fitting 4 folds for each of 300 candidates, totalling 1200 fits
[CV 1/4] END C=0.1, degree=2, gamma=scale, kernel=linear;, score=0.492 total time=   0.0s
[CV 2/4] END C=0.1, degree=2, gamma=scale, kernel=linear;, score=0.354 total time=   0.0s
[CV 3/4] END C=0.1, degree=2, gamma=scale, kernel=linear;, score=0.415 total time=   0.0s
[CV 4/4] END C=0.1, degree=2, gamma=scale, kernel=linear;, score=0.422 total time=   0.0s
[CV 1/4] END C=0.1, degree=2, gamma=scale, kernel=poly;, score=0.108 total time=   0.0s
[CV 2/4] END C=0.1, degree=2, gamma=scale, kernel=poly;, score=0.138 total time=   0.0s
[CV 3/4] END C=0.1, degree=2, gamma=scale, kernel=poly;, score=0.138 total time=   0.0s
[CV 4/4] END C=0.1, degree=2, gamma=scale, kernel=poly;, score=0.141 total time=   0.0s
[CV 1/4] END C=0.1, degree=2, gamma=scale, kernel=rbf;, score=0.108 total time=   0.0s
[CV 2/4] END C=0.1, degree=2, gamma=scale, kernel=rbf;, score=0.138 total time=   0.0s
[CV 3/4] END C=0.1, degree=2, gamma=scale, kernel=

[CV 3/4] END C=0.1, degree=3, gamma=auto, kernel=poly;, score=0.754 total time=   0.0s
[CV 4/4] END C=0.1, degree=3, gamma=auto, kernel=poly;, score=0.766 total time=   0.0s
[CV 1/4] END C=0.1, degree=3, gamma=auto, kernel=rbf;, score=0.108 total time=   0.0s
[CV 2/4] END C=0.1, degree=3, gamma=auto, kernel=rbf;, score=0.138 total time=   0.0s
[CV 3/4] END C=0.1, degree=3, gamma=auto, kernel=rbf;, score=0.138 total time=   0.0s
[CV 4/4] END C=0.1, degree=3, gamma=auto, kernel=rbf;, score=0.141 total time=   0.0s
[CV 1/4] END C=0.1, degree=3, gamma=auto, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 2/4] END C=0.1, degree=3, gamma=auto, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 3/4] END C=0.1, degree=3, gamma=auto, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 4/4] END C=0.1, degree=3, gamma=auto, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 1/4] END C=0.1, degree=3, gamma=0.1, kernel=linear;, score=0.492 total time=   0.0s
[CV 2/4] END C=0.1, degree=3, gamm

[CV 1/4] END C=0.1, degree=4, gamma=0.1, kernel=rbf;, score=0.108 total time=   0.0s
[CV 2/4] END C=0.1, degree=4, gamma=0.1, kernel=rbf;, score=0.138 total time=   0.0s
[CV 3/4] END C=0.1, degree=4, gamma=0.1, kernel=rbf;, score=0.138 total time=   0.0s
[CV 4/4] END C=0.1, degree=4, gamma=0.1, kernel=rbf;, score=0.141 total time=   0.0s
[CV 1/4] END C=0.1, degree=4, gamma=0.1, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 2/4] END C=0.1, degree=4, gamma=0.1, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 3/4] END C=0.1, degree=4, gamma=0.1, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 4/4] END C=0.1, degree=4, gamma=0.1, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 1/4] END C=0.1, degree=4, gamma=1, kernel=linear;, score=0.492 total time=   0.0s
[CV 2/4] END C=0.1, degree=4, gamma=1, kernel=linear;, score=0.354 total time=   0.0s
[CV 3/4] END C=0.1, degree=4, gamma=1, kernel=linear;, score=0.415 total time=   0.0s
[CV 4/4] END C=0.1, degree=4, gamma=1, kernel=

[CV 4/4] END C=1, degree=2, gamma=1, kernel=rbf;, score=0.766 total time=   0.0s
[CV 1/4] END C=1, degree=2, gamma=1, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 2/4] END C=1, degree=2, gamma=1, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 3/4] END C=1, degree=2, gamma=1, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 4/4] END C=1, degree=2, gamma=1, kernel=sigmoid;, score=0.016 total time=   0.0s
[CV 1/4] END C=1, degree=2, gamma=10, kernel=linear;, score=0.769 total time=   0.0s
[CV 2/4] END C=1, degree=2, gamma=10, kernel=linear;, score=0.723 total time=   0.0s
[CV 3/4] END C=1, degree=2, gamma=10, kernel=linear;, score=0.754 total time=   0.0s
[CV 4/4] END C=1, degree=2, gamma=10, kernel=linear;, score=0.766 total time=   0.0s
[CV 1/4] END C=1, degree=2, gamma=10, kernel=poly;, score=0.769 total time=   0.0s
[CV 2/4] END C=1, degree=2, gamma=10, kernel=poly;, score=0.723 total time=   0.0s
[CV 3/4] END C=1, degree=2, gamma=10, kernel=poly;, score=0.754 total tim

[CV 1/4] END C=1, degree=4, gamma=scale, kernel=rbf;, score=0.323 total time=   0.0s
[CV 2/4] END C=1, degree=4, gamma=scale, kernel=rbf;, score=0.200 total time=   0.0s
[CV 3/4] END C=1, degree=4, gamma=scale, kernel=rbf;, score=0.292 total time=   0.0s
[CV 4/4] END C=1, degree=4, gamma=scale, kernel=rbf;, score=0.297 total time=   0.0s
[CV 1/4] END C=1, degree=4, gamma=scale, kernel=sigmoid;, score=0.077 total time=   0.0s
[CV 2/4] END C=1, degree=4, gamma=scale, kernel=sigmoid;, score=0.062 total time=   0.0s
[CV 3/4] END C=1, degree=4, gamma=scale, kernel=sigmoid;, score=0.138 total time=   0.0s
[CV 4/4] END C=1, degree=4, gamma=scale, kernel=sigmoid;, score=0.109 total time=   0.0s
[CV 1/4] END C=1, degree=4, gamma=auto, kernel=linear;, score=0.769 total time=   0.0s
[CV 2/4] END C=1, degree=4, gamma=auto, kernel=linear;, score=0.723 total time=   0.0s
[CV 3/4] END C=1, degree=4, gamma=auto, kernel=linear;, score=0.754 total time=   0.0s
[CV 4/4] END C=1, degree=4, gamma=auto, ker

...

[CV 3/4] END C=10, degree=4, gamma=10, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 4/4] END C=10, degree=4, gamma=10, kernel=sigmoid;, score=0.016 total time=   0.0s
[CV 1/4] END C=100, degree=2, gamma=scale, kernel=linear;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=2, gamma=scale, kernel=linear;, score=0.723 total time=   0.0s
[CV 3/4] END C=100, degree=2, gamma=scale, kernel=linear;, score=0.754 total time=   0.0s
[CV 4/4] END C=100, degree=2, gamma=scale, kernel=linear;, score=0.766 total time=   0.0s
[CV 1/4] END C=100, degree=2, gamma=scale, kernel=poly;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=2, gamma=scale, kernel=poly;, score=0.723 total time=   0.0s
[CV 3/4] END C=100, degree=2, gamma=scale, kernel=poly;, score=0.754 total time=   0.0s
[CV 4/4] END C=100, degree=2, gamma=scale, kernel=poly;, score=0.766 total time=   0.0s
[CV 1/4] END C=100, degree=2, gamma=scale, kernel=rbf;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=

[CV 1/4] END C=100, degree=3, gamma=auto, kernel=sigmoid;, score=0.092 total time=   0.0s
[CV 2/4] END C=100, degree=3, gamma=auto, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 3/4] END C=100, degree=3, gamma=auto, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 4/4] END C=100, degree=3, gamma=auto, kernel=sigmoid;, score=0.016 total time=   0.0s
[CV 1/4] END C=100, degree=3, gamma=0.1, kernel=linear;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=3, gamma=0.1, kernel=linear;, score=0.723 total time=   0.0s
[CV 3/4] END C=100, degree=3, gamma=0.1, kernel=linear;, score=0.754 total time=   0.0s
[CV 4/4] END C=100, degree=3, gamma=0.1, kernel=linear;, score=0.766 total time=   0.0s
[CV 1/4] END C=100, degree=3, gamma=0.1, kernel=poly;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=3, gamma=0.1, kernel=poly;, score=0.723 total time=   0.0s
[CV 3/4] END C=100, degree=3, gamma=0.1, kernel=poly;, score=0.754 total time=   0.0s
[CV 4/4] END C=100, degree=3, 

[CV 1/4] END C=100, degree=4, gamma=1, kernel=poly;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=4, gamma=1, kernel=poly;, score=0.723 total time=   0.0s
[CV 3/4] END C=100, degree=4, gamma=1, kernel=poly;, score=0.754 total time=   0.0s
[CV 4/4] END C=100, degree=4, gamma=1, kernel=poly;, score=0.766 total time=   0.0s
[CV 1/4] END C=100, degree=4, gamma=1, kernel=rbf;, score=0.769 total time=   0.0s
[CV 2/4] END C=100, degree=4, gamma=1, kernel=rbf;, score=0.723 total time=   0.0s
[CV 3/4] END C=100, degree=4, gamma=1, kernel=rbf;, score=0.754 total time=   0.0s
[CV 4/4] END C=100, degree=4, gamma=1, kernel=rbf;, score=0.766 total time=   0.0s
[CV 1/4] END C=100, degree=4, gamma=1, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 2/4] END C=100, degree=4, gamma=1, kernel=sigmoid;, score=0.000 total time=   0.0s
[CV 3/4] END C=100, degree=4, gamma=1, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 4/4] END C=100, degree=4, gamma=1, kernel=sigmoid;, score=0.016 tot

[CV 2/4] END C=1000, degree=2, gamma=10, kernel=rbf;, score=0.723 total time=   0.0s
[CV 3/4] END C=1000, degree=2, gamma=10, kernel=rbf;, score=0.754 total time=   0.0s
[CV 4/4] END C=1000, degree=2, gamma=10, kernel=rbf;, score=0.766 total time=   0.0s
[CV 1/4] END C=1000, degree=2, gamma=10, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 2/4] END C=1000, degree=2, gamma=10, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 3/4] END C=1000, degree=2, gamma=10, kernel=sigmoid;, score=0.015 total time=   0.0s
[CV 4/4] END C=1000, degree=2, gamma=10, kernel=sigmoid;, score=0.016 total time=   0.0s
[CV 1/4] END C=1000, degree=3, gamma=scale, kernel=linear;, score=0.769 total time=   0.0s
[CV 2/4] END C=1000, degree=3, gamma=scale, kernel=linear;, score=0.723 total time=   0.0s
[CV 3/4] END C=1000, degree=3, gamma=scale, kernel=linear;, score=0.754 total time=   0.0s
[CV 4/4] END C=1000, degree=3, gamma=scale, kernel=linear;, score=0.766 total time=   0.0s
[CV 1/4] END C=1000, degr

[CV 1/4] END C=1000, degree=4, gamma=scale, kernel=rbf;, score=0.769 total time=   0.0s
[CV 2/4] END C=1000, degree=4, gamma=scale, kernel=rbf;, score=0.723 total time=   0.0s
[CV 3/4] END C=1000, degree=4, gamma=scale, kernel=rbf;, score=0.754 total time=   0.0s
[CV 4/4] END C=1000, degree=4, gamma=scale, kernel=rbf;, score=0.766 total time=   0.0s
[CV 1/4] END C=1000, degree=4, gamma=scale, kernel=sigmoid;, score=0.262 total time=   0.0s
[CV 2/4] END C=1000, degree=4, gamma=scale, kernel=sigmoid;, score=0.215 total time=   0.0s
[CV 3/4] END C=1000, degree=4, gamma=scale, kernel=sigmoid;, score=0.231 total time=   0.0s
[CV 4/4] END C=1000, degree=4, gamma=scale, kernel=sigmoid;, score=0.234 total time=   0.0s
[CV 1/4] END C=1000, degree=4, gamma=auto, kernel=linear;, score=0.769 total time=   0.0s
[CV 2/4] END C=1000, degree=4, gamma=auto, kernel=linear;, score=0.723 total time=   0.0s
[CV 3/4] END C=1000, degree=4, gamma=auto, kernel=linear;, score=0.754 total time=   0.0s
[CV 4/4] E

In [26]:
# GridSearchCV result
print("Best: {} using {}".format(round(grid_result.best_score_, 6), grid_result.best_params_)) 

means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.752945 using {'C': 0.1, 'degree': 2, 'gamma': 1, 'kernel': 'poly'}
0.420853 (0.049058) with: {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'linear'}
0.131310 (0.013664) with: {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'poly'}
0.131310 (0.013664) with: {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'rbf'}
0.096575 (0.029716) with: {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'sigmoid'}
0.420853 (0.049058) with: {'C': 0.1, 'degree': 2, 'gamma': 'auto', 'kernel': 'linear'}
0.444291 (0.060242) with: {'C': 0.1, 'degree': 2, 'gamma': 'auto', 'kernel': 'poly'}
0.131310 (0.013664) with: {'C': 0.1, 'degree': 2, 'gamma': 'auto', 'kernel': 'rbf'}
0.000000 (0.000000) with: {'C': 0.1, 'degree': 2, 'gamma': 'auto', 'kernel': 'sigmoid'}
0.420853 (0.049058) with: {'C': 0.1, 'degree': 2, 'gamma': 0.1, 'kernel': 'linear'}
0.254868 (0.055105) with: {'C': 0.1, 'degree': 2, 'gamma': 0.1, 'kernel': 'poly'}
0.131310 (0.013664) with: {'C': 0.1, 'degree': 2, 'gamma': 0.1, 'ke

In [27]:
# Support Vector Machine
svm = SVC(C=0.1, degree=2, gamma= 1, kernel='poly', random_state=1)
svm_cv_results = cross_val_score(svm, X_train, y_train, cv=4, verbose=3)

[CV] END ................................ score: (test=0.769) total time=   0.0s
[CV] END ................................ score: (test=0.723) total time=   0.0s
[CV] END ................................ score: (test=0.754) total time=   0.0s
[CV] END ................................ score: (test=0.766) total time=   0.0s


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s finished


In [28]:
print("accuracy: {}, std: {}".format(svm_cv_results.mean(), svm_cv_results.std()))

accuracy: 0.7529447115384615, std: 0.018158488211042722


## Random Forest

In [29]:
# number of trees in random forest
n_estimators = [1, 2, 4, 8, 16, 32]
# split criterion
criterion = ["gini", "entropy"]
# maximum number of levels in tree
max_depth = [5, 10, 15, 20]
# max considered features per split
max_features = ["sqrt", "log2", None]
# min samples per leaf node
min_samples_leaf = [1, 2, 3, 4, 5]
# create the params grid
param_grid = {'n_estimators': n_estimators,
              'criterion': criterion,
               'max_depth': max_depth,
             'max_features': max_features,
             'min_samples_leaf': min_samples_leaf}

In [30]:
# GridSearchCV fit
model = RandomForestClassifier(n_jobs=4, random_state=1)

grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring="accuracy", cv=4, verbose=3)
grid_result = grid.fit(X_train.values, y_train.values)

Fitting 4 folds for each of 720 candidates, totalling 2880 fits
[CV 1/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=1;, score=0.662 total time=   2.3s
[CV 2/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=1;, score=0.492 total time=   1.2s
[CV 3/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=1;, score=0.523 total time=   1.4s
[CV 4/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=1;, score=0.516 total time=   1.2s
[CV 1/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=2;, score=0.785 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=2;, score=0.615 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=1, n_estimators=2;, score=0.662 total time=   0.0s
[CV 4/4] END criterion=gini, max_

[CV 3/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=3, n_estimators=16;, score=0.754 total time=   0.0s
[CV 4/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=3, n_estimators=16;, score=0.750 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=3, n_estimators=32;, score=0.769 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=3, n_estimators=32;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=3, n_estimators=32;, score=0.769 total time=   0.0s
[CV 4/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=3, n_estimators=32;, score=0.781 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=4, n_estimators=1;, score=0.338 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=sqrt, min_samples_leaf=4, n_estimato

[CV 4/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=8;, score=0.797 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=16;, score=0.785 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=16;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=16;, score=0.754 total time=   0.0s
[CV 4/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=16;, score=0.797 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=32;, score=0.769 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimators=32;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=1, n_estimato

[CV 4/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=4;, score=0.703 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=8;, score=0.723 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=8;, score=0.692 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=8;, score=0.723 total time=   0.0s
[CV 4/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=8;, score=0.719 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=16;, score=0.785 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=16;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=log2, min_samples_leaf=4, n_estimators=1

[CV 3/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=2;, score=0.615 total time=   0.0s
[CV 4/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=2;, score=0.750 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=4;, score=0.769 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=4;, score=0.692 total time=   0.0s
[CV 3/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=4;, score=0.631 total time=   0.0s
[CV 4/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=4;, score=0.750 total time=   0.0s
[CV 1/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=8;, score=0.769 total time=   0.0s
[CV 2/4] END criterion=gini, max_depth=5, max_features=None, min_samples_leaf=2, n_estimators=8;,

...

[CV 1/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=8;, score=0.769 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=8;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=8;, score=0.754 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=8;, score=0.797 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=16;, score=0.785 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=16;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=16;, score=0.754 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=sqrt, m

[CV 1/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=4;, score=0.538 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=4;, score=0.585 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=4;, score=0.646 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=4;, score=0.641 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=8;, score=0.615 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=8;, score=0.631 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_samples_leaf=5, n_estimators=8;, score=0.600 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=sqrt, min_

[CV 2/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=1;, score=0.585 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=1;, score=0.385 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=1;, score=0.562 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=2;, score=0.723 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=2;, score=0.692 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=2;, score=0.631 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=3, n_estimators=2;, score=0.656 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=log2, min_

[CV 2/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=16;, score=0.662 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=16;, score=0.692 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=16;, score=0.688 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=32;, score=0.754 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=32;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=32;, score=0.769 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=log2, min_samples_leaf=5, n_estimators=32;, score=0.766 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=Non

[CV 4/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=4;, score=0.750 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=8;, score=0.754 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=8;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=8;, score=0.738 total time=   0.0s
[CV 4/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=8;, score=0.766 total time=   0.0s
[CV 1/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=16;, score=0.769 total time=   0.0s
[CV 2/4] END criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=3, n_estimators=16;, score=0.723 total time=   0.0s
[CV 3/4] END criterion=entropy, max_depth=20, max_features=None, mi

In [31]:
# GridSearchCV result
print("Best: {} using {}".format(round(grid_result.best_score_, 6), grid_result.best_params_)) 

means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.776202 using {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 2}
0.548137 (0.066449) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 1}
0.714603 (0.077991) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 2}
0.741406 (0.026615) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 4}
0.768510 (0.033443) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 8}
0.764603 (0.028645) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 16}
0.764603 (0.026499) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 32}
0.524820 (0.080539) with: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min

In [32]:
# Random Forest
random_forest = RandomForestClassifier(criterion='gini', max_depth=10, max_features='sqrt', min_samples_leaf= 1, n_estimators=2, n_jobs=4, random_state=1)
random_forest_cv_results = cross_val_score(random_forest, X_train, y_train, cv=4, verbose=3)

[CV] END ................................ score: (test=0.800) total time=   0.0s
[CV] END ................................ score: (test=0.738) total time=   0.0s
[CV] END ................................ score: (test=0.754) total time=   0.0s
[CV] END ................................ score: (test=0.812) total time=   0.0s


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s finished


In [33]:
print("accuracy: {}, std: {}".format(random_forest_cv_results.mean(), random_forest_cv_results.std()))

accuracy: 0.776201923076923, std: 0.030854561665849607


## Neural Network

In [34]:
# size of hidden layer
hidden_layer_sizes = [(9,), (10,), (11,), (12,), (13,), (14,), (15,), (40,)]
# hidden layer activation function
activation = ['identity', 'logistic', 'tanh', 'relu']
# solver
solver = ['lbfgs']
# L2 regularization term strength
alpha = [0.0001, 0.001, 0.01, 0.1, 1]
# create the params grid
param_grid = {'hidden_layer_sizes': hidden_layer_sizes,
              'activation': activation,
              'solver': solver,
              'alpha': alpha}

In [35]:
# GridSearchCV fit
model = MLPClassifier(random_state=1)

grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring="accuracy", cv=4, verbose=3)
grid_result = grid.fit(X_train.values, y_train.values)

Fitting 4 folds for each of 160 candidates, totalling 640 fits


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[CV 1/4] END activation=identity, alpha=0.0001, hidden_layer_sizes=(9,), solver=lbfgs;, score=0.769 total time=   0.1s


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[CV 2/4] END activation=identity, alpha=0.0001, hidden_layer_sizes=(9,), solver=lbfgs;, score=0.723 total time=   0.1s


...

C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[CV 3/4] END activation=relu, alpha=1, hidden_layer_sizes=(40,), solver=lbfgs;, score=0.754 total time=   0.4s


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[CV 4/4] END activation=relu, alpha=1, hidden_layer_sizes=(40,), solver=lbfgs;, score=0.766 total time=   0.4s


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [36]:
# GridSearchCV result
print("Best: {} using {}".format(round(grid_result.best_score_, 6), grid_result.best_params_)) 

means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.764603 using {'activation': 'relu', 'alpha': 0.1, 'hidden_layer_sizes': (13,), 'solver': 'lbfgs'}
0.760757 (0.026664) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (9,), 'solver': 'lbfgs'}
0.752945 (0.018158) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (10,), 'solver': 'lbfgs'}
0.752945 (0.018158) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (11,), 'solver': 'lbfgs'}
0.760757 (0.026664) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (12,), 'solver': 'lbfgs'}
0.752945 (0.018158) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (13,), 'solver': 'lbfgs'}
0.760757 (0.026664) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (14,), 'solver': 'lbfgs'}
0.760757 (0.026664) with: {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (15,), 'solver': 'lbfgs'}
0.760757 (0.026664) with: {'activation': 'identity', 'alpha': 0.0001,

In [37]:
# neural network
neural_network = MLPClassifier(activation='relu', alpha=0.1, hidden_layer_sizes=(13,), solver='lbfgs', random_state=1)
neural_network_cv_results = cross_val_score(neural_network, X_train, y_train, cv=4, verbose=3)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s remaining:    0.0s


[CV] END ................................ score: (test=0.785) total time=   0.2s


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.5s remaining:    0.0s


[CV] END ................................ score: (test=0.723) total time=   0.2s


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[CV] END ................................ score: (test=0.754) total time=   0.2s
[CV] END ................................ score: (test=0.797) total time=   0.2s


C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\neural_network\_multilayer_perceptron.py:559: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    1.1s finished


In [38]:
print("accuracy: {}, std: {}".format(neural_network_cv_results.mean(), neural_network_cv_results.std()))

accuracy: 0.7646033653846154, std: 0.028644817704337362


In [39]:
# print each model accuracy (std)
print("svm(svc): {} ({})".format(svm_cv_results.mean(), svm_cv_results.std()))
print("random forest: {} ({})".format(random_forest_cv_results.mean(), random_forest_cv_results.std()))
print("neural network: {} ({})".format(neural_network_cv_results.mean(), neural_network_cv_results.std()))

svm(svc): 0.7529447115384615 (0.018158488211042722)
random forest: 0.776201923076923 (0.030854561665849607)
neural network: 0.7646033653846154 (0.028644817704337362)


# Create Model

In [40]:
# fit final model
final_model = RandomForestClassifier(criterion='gini', max_depth=10, max_features='sqrt', min_samples_leaf= 1, n_estimators=2, n_jobs=4, random_state=1)
final_model.fit(X_train.values, y_train.values)

y_pred = final_model.predict(X_test.values)

In [41]:
# train test accuracy score
print("train accuracy: {}, test accuracy: {}".format(round(final_model.score(X_train.values, y_train.values), 3), round(final_model.score(X_test.values, y_test.values), 3)))

train accuracy: 0.78, test accuracy: 0.793


In [42]:
# classification report
print(classification_report(y_test.values, y_pred, digits=3))

              precision    recall  f1-score   support

           0      1.000     1.000     1.000         3
           1      0.000     0.000     0.000         3
           2      1.000     1.000     1.000         3
           3      1.000     1.000     1.000         3
           4      1.000     1.000     1.000         3
           5      0.000     0.000     0.000         3
           6      0.000     0.000     0.000         3
           7      1.000     0.333     0.500         3
           8      1.000     1.000     1.000         3
           9      0.333     1.000     0.500         3
          10      1.000     1.000     1.000         3
          11      0.333     1.000     0.500         3
          12      0.000     0.000     0.000         3
          13      1.000     1.000     1.000         3
          14      1.000     1.000     1.000         3
          15      0.000     0.000     0.000         3
          16      1.000     1.000     1.000         3
          17      1.000    

C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\metrics\_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\metrics\_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\metrics\_classification.py:1327: Unde

In [43]:
# top 5 accuracy
y_pred_probs = final_model.predict_proba(X_test.values)
print('top 5 accuracy: %s' % top_k_accuracy_score(y_test.values, y_pred_probs, k=5))

top 5 accuracy: 1.0


# Prediction Testing

final_model.predict testing

In [44]:
final_prediction = final_model.predict([[6,1,1,1,0,0,1,0]])
final_prediction

array([28])

In [45]:
# inverse transform predicted class
print(lbl_encoder.inverse_transform(final_prediction)[0])

Discoaster pseudovariabilis


final_model.predict_proba testing

In [46]:
probs = final_model.predict_proba([[6,1,1,1,0,0,1,0]])
probs

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.07255747, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.28017241, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.37715517, 0.        ,
        0.        , 0.27011494, 0.        , 0.        , 0.        ,
        0.        , 0.        ]])

In [47]:
# sort prediction probabilitises
sorted_probs = sorted(probs[0], reverse=True)
sorted_probs

[0.3771551724137931,
 0.2801724137931034,
 0.27011494252873564,
 0.07255747126436782,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0]

In [48]:
# search top five prediction probabilities
top_five_probs = [sorted_probs[0], sorted_probs[1], sorted_probs[2], sorted_probs[3], sorted_probs[4]]
top_five_probs

[0.3771551724137931,
 0.2801724137931034,
 0.27011494252873564,
 0.07255747126436782,
 0.0]

In [49]:
# search best five prediction class
best_n = np.argsort(probs)[:,-5:]
best_n[0]

array([ 8,  7, 31, 15, 28], dtype=int64)

In [50]:
# rank best five prediction class from the best
best_five_species = [best_n[0][4], best_n[0][3], best_n[0][2], best_n[0][1], best_n[0][0]]
best_five_species

[28, 15, 31, 7, 8]

In [51]:
# inverse transform ranked best five predicted class
lbl_encoder.inverse_transform(best_five_species)

array(['Discoaster pseudovariabilis', 'Discoaster exilis',
       'Discoaster subsurculus', 'Discoaster bollii',
       'Discoaster braarudii'], dtype=object)

# Saving Models

In [52]:
# save trained random forest model
save_model(final_model, "random_forest_nannomate.pkl")

Model is pickled as model/random_forest_nannomate.pkl


In [53]:
# save label encoder
save_model(lbl_encoder, "lbl_encoder_nannomate.pkl")

Model is pickled as model/lbl_encoder_nannomate.pkl


In [54]:
# save ordinal encoder
save_model(ordl_encoder, "ordl_encoder_nannomate.pkl")

Model is pickled as model/ordl_encoder_nannomate.pkl
